<a href="https://colab.research.google.com/github/Dijah-Tunrayo/lab-4-llm-decision-support/blob/main/lab-4-llm-decision-support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key = API_KEY)

MODEL = "gemini-flash-lite-latest"

print("Client ready.")

Client ready.


In [ ]:
import time

def ask_llm_retry(prompt, temperature=0.7, max_tokens=500, retries=3):
    for attempt in range(retries):
        try:
            return ask_llm(prompt, temperature=temperature, max_tokens=max_tokens)
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retry {attempt+1}/{retries}... ({e})")
                time.sleep(5)
            else:
                raise

**PART 1.1**

In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?


#================================================================================================================#
# TODO 1: Helper function to be reused for the whole Lab


def ask_llm(user_prompt, system_prompt = "You are a helpful assistant.",
            temperature = 0.7, max_tokens = 500):
   response = client.models.generate_content(
       model = MODEL,
       contents = user_prompt,
       config = {
           "system_instruction": system_prompt,
           "temperature": temperature,
           "max_output_tokens": max_tokens,
       }
   )
   return response.text
#================================================================================================================#

In [ ]:
#================================================================================================================#
answer = ask_llm("What is 2 + 2?")
print(answer)
#================================================================================================================#

2 + 2 = 4


In [ ]:
#================================================================================================================#
response = client.models.generate_content(
    model = MODEL,
    contents = "What is 2 + 2?"
)
print(response.usage_metadata)
#================================================================================================================#

cache_tokens_details=None cached_content_token_count=None candidates_token_count=7 candidates_tokens_details=None prompt_token_count=9 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=9
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=16 traffic_type=None


After running the response.usage_metadata, the call consumed 85 tokens total:

1. 8 to 9 for the input prompt
2. 7 tokens for the visible answer text
3. 65 to 70 invisible internal thinking tokens

**STUDENT REASONING**

1. The difference between system and user roles is that, system roles, set the behaviour, persona and constraints for the model. It serves as something like a job description for the model before any actual work is started. An example could be: In a loan application system, the system prompt might say "You are a meticulous assistant to a very demanding microfinance loan officer. Be extremely factual, never invent information and make sure to be precise and concise."
The user prompt on the other hand contains the specific request or the content to be processed by the model. For example "Summarise this loan application letter and fix grammatical errors:[~then the text (letter) goes here].


2. A token, is roughly a word unit or a sub-word unit which is often about 4 English characters, for example "microfinance" might be about 2 to 3 tokens.
The reason API providers bill per token for both input and output is because, it directly reflects computational cost. The longer the input the more the processing is done and the longer the output, the more generation steps are occuring. A per request billing will be unfair and charge the same for a 40 word request and a 2000 word document.



**PART 1.2**

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
#================================================================================================================#
# TODO 1
question = "Suggest a name for a savings product for market traders in Accra"

print("=====TEMPERATURE = 0.0=====")
for i in range(5):
  print(f"{i+1}: {ask_llm(question, temperature = 0.0)}") # TODO 2

print()
print("=====TEMPERATURE = 1.2=====")
for i in range(5):
  print(f"{i+1}: {ask_llm(question, temperature = 1.2)}") # TODO 2
#================================================================================================================#

=====TEMPERATURE = 0.0=====
1: Here are several savings product name suggestions for market traders in Accra, broken down by the "vibe" or marketing angle that best fits your target audience. 

Many traders in markets like Makola, Kaneshie, or Agbogbloshie respond well to names that emphasize growth, respect for their hustle, and community.

### 1. High-Energy & Motivational (Pidgin & Twi Blend)
*These names resonate with the daily hustle, resilience, and ambition of traders.*
* **Sikaɛni Plus:** (A play on "Sika" [Money] and "Okaeini" [Owner/Person of substance]) – Means "The Money Maker's Account."
* **Bɔkɔɔ Savings:** ("Bɔkɔɔ" means slowly/peacefully in Twi) – Great for a product aimed at steady, stress-free daily contributions (susu style).
* **Nkabom Trader Fund:** ("Nkabom" means unity/bringing together) – Appeals to market association groups.
* **Hustle & Harvest:** Modern, aspirational, and speaks directly to putting in work and reaping the rewards.

### 2. Traditional & Trustw

**STUDENT REASONING**

1. At temperature = 0.0, all the outputs look heavily structured the same way, with similar section headers that are Twi, Ga and Pidgin English inspired, with themes of growth and personalised empowerment, with some of these names (Sika Dwa, Nkabom) appearing over and over again. At temperature = 1.2, however, though the structures are similar, the creativity is noticeably much more, and there's variation in the names and themes used.

2. For the loan decision-support system, temperature being = 0.0 (signifying a very low temp) is most appropriate because the system must and needs to be deterministic and reliable, since a loan officer needs consistent and factual output and not creative variations (a trait of increased temperatures.)

**PART 2.0**

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


**PART 3.1**

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

#================================================================================================================#
# TODO 1
SUMMARY_PROMPT_V1 = "Summarize this: {letter_text}"

print("=====V1 on L002=====")
print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text = LETTERS["L002"])))

print()
print("=====V1 on L006=====")
print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text = LETTERS["L006"])))

#================================================================================================================#

=====V1 on L002=====
**Summary:** 

Kwame Boateng, a commercial driver in Kumasi, is urgently requesting a GHS 25,000 loan without collateral to repair his trotro engine and pay personal debts. He plans to repay the amount after the festive season when business improves.

=====V1 on L006=====
**Summary:**

Kofi, a 22-year-old with no prior business experience, is requesting a GHS 50,000 unsecured loan to simultaneously start a car wash, a provision shop, and a phone importation business from Dubai. He relies on friends' validation of his business mindset and his own trustworthiness, promising to repay the loan in one year once his businesses succeed.


In [ ]:
#================================================================================================================#
# TODO 2
SUMMARY_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer. "
    "Summarize loan applications factually and neutrally in 3-4 sentences. "
    "Do not invent details not stated in the letter, and do not add your own "
    "interpretation or judgment of the applicant."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

print("=====V2 on L002=====")
print(ask_llm(SUMMARY_PROMPT_V2.format(letter_text = LETTERS["L002"]), system_prompt = SUMMARY_SYSTEM_PROMPT, temperature = 0))


print()
print("=====V2 on L006=====")
print(ask_llm(SUMMARY_PROMPT_V2.format(letter_text = LETTERS["L006"]), system_prompt = SUMMARY_SYSTEM_PROMPT, temperature = 0))

#================================================================================================================#

=====V2 on L002=====
Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000 to repair a trotro engine and settle personal debts. He states that business has been slow but anticipates an increase following the festive season, though he has not provided a specific repayment schedule. The applicant currently has no collateral to offer for the loan.

=====V2 on L006=====
Kofi, a 22-year-old applicant, is requesting a loan of GHS 50,000 to start a car washing business, open a provision shop, and import phones from Dubai. He has not yet started any of these businesses and currently has no collateral to offer. He states that he is trustworthy and plans to repay the loan in one year once his businesses are booming.


**STUDENT REASONING**
1. V1 produced generic summaries that introduces interpretive language and missed key financial details. It said business was slow 'due to the festive season', a claim not in the letter. V2 clearly states the fact that the letter said it will improve after the season and doesnt blame the season for the slowdown, which V1 did.
Secondly, while V1 commented about Kofi "relying on friends' opinions for his business acumen", V2 corrects this by neutrally reporting that "his friends consider him business-minded".


2. 'No invented details' is essential because loan officers will have to make decisions based on the generated summaries. If the modeol invents a repayment period or collateral or situation that mustn't exist in the letter, or even requesting some form of documentation that an applicant never claimed to have or give. This failure mode is known as hallucination, and it occurs when an LLM generates plausible sounding but false, factually incorrect information. In the situation as high stakes as loaning, lending, etc., hallucination stays very dangerous.  

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

#================================================================================================================#
# TODO 1
EXAMPLE_LETTER = """
Dear Sir,
My name is Ama Kwofie, I sell fabrics at Kantamanto Market.
I am requesting GHS 5,000 to restock ahead of Easter.
I make about GHS 600 profit monthly.
Repayment will be GHS 300 per month for 18 months.
My brother will guarantee the loan with his shop.
"""
EXAMPLE_JSON = """
{
"applicant_name": "Ama Kwofie",
"amount_ghs": 5000,
"purpose": "restock fabric ahead of Easter",
"monthly_profit_ghs": 600,
"has_collateral_or_guarantor": true,
"repayment_months": 18
}
"""

EXTRACT_SYSTEM_PROMPT = (
"""
You are a precise data extraction assistant.
Extract loan application fields as a single valid JSON object with EXACTLY these key:
applicant_name (string), amount_ghs (number), purpose (string), monthly_profit_ghs (number or null),
has_collateral_or_guarantor (boolean),
repayment_months (number or null).
If a field is not stated in the letter, use null.
Do not guess.
Return ONLY the JSON object, no other text, no markdown fences.
"""
)

EXTRACT_PROMPT = f"""
Here is an example.

Letter: {EXAMPLE_LETTER}
JSON: {EXAMPLE_JSON}

Now extract the same fields from this letter;

LETTER:
<<LETTER>>

JSON:
"""
#================================================================================================================#

In [ ]:
#================================================================================================================#
# TODO 2
import json

def extract_fields(letter_text):
  prompt = EXTRACT_PROMPT.replace("<<LETTER>>", letter_text)
  raw_output = ask_llm(prompt, system_prompt = EXTRACT_SYSTEM_PROMPT, temperature = 0)

  # Strips fences if added anyway
  cleaned = raw_output.strip()
  if cleaned.startswith("'''"):
    cleaned = cleaned.strip("'")
    cleaned = cleaned.replace("json", "", 1).strip()
  try:
    return json.loads(cleaned)
  except json.JSONDecodeError:
    print(f"Failed to parse JSON:\n{raw_output}")
    return None
#================================================================================================================#

In [ ]:
#================================================================================================================#
# TODO 3
import pandas as pd

results = []

for letter_id, letter_text in LETTERS.items():
  fields = extract_fields(letter_text)
  if fields:
    fields["letter_id"] = letter_id
    results.append(fields)

df = pd.DataFrame(results)
df = df[["letter_id"] + [c for c in df.columns if c != "letter_id"]]
df


#================================================================================================================#

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle some personal ...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,feed and 500 new layers for poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**STUDENT REASONING**

1. If we use one of the actual test etters as a few-shot example would "leak" the correct answer format, or even values to the model, thereby inflating the performance artificially. This is similar to studying for an exam using that same exam's marking scheme. The few-shot example must come from an independent, handwritten example so we test whether the model can generalise unseen letters, and not whether it can copy from the example.


2. Without the 'use null, do not guess' instruction, the model would often invent plausible values for missing fields. I tested this by removing the instruction and rerunning it on L002(whose letter doesn't mention monthly profit):

In [ ]:
NO_NULL_SYSTEM_PROMPT = EXTRACT_SYSTEM_PROMPT.replace(
    "If a field is not stated in the letter, use null. \nDo not guess.\n", ""
)

print(NO_NULL_SYSTEM_PROMPT)  # sanity check — confirm the instruction is actually gone


You are a precise data extraction assistant.
Extract loan application fields as a single valid JSON object with EXACTLY these key:
applicant_name (string), amount_ghs (number), purpose (string), monthly_profit_ghs (number or null),
has_collateral_or_guarantor (boolean),
repayment_months (number or null).
If a field is not stated in the letter, use null.
Do not guess.
Return ONLY the JSON object, no other text, no markdown fences.



In [ ]:
test_prompt = EXTRACT_PROMPT.replace("<<LETTER>>", LETTERS["L002"])
test_output = ask_llm(test_prompt, system_prompt=NO_NULL_SYSTEM_PROMPT, temperature=0)
print(test_output)

{
"applicant_name": "Kwame Boateng",
"amount_ghs": 25000,
"purpose": "repair trotro engine and settle some personal debts",
"monthly_profit_ghs": null,
"has_collateral_or_guarantor": false,
"repayment_months": null
}


Though the model correctly returned null for the monthly_profit_ghs even without the instruction (a suggestion that Gemini's Flash has an inbuilt caution against fabricating unstated numeric fields, for this case). It is imperative that the instruction is explicitly stated to serve as a safeguard since LLM behaviour isnt guaranteed to be consistent acriss different models, letters, temperature settings among others, and there is no protection against the model inventing or hallucinating values. Given how costly and dangerous a fabricated financial figure would be in a lending context, relying on implicit model behavior instead of an explicit constraints would be a risky design choice.


3. Extraction has a need and requirement for accuracy and consistency. It has one correct answer, and the model must always produce it. Setting the temperature = 0 makes the model a deterministic one, having it always selting the most likely token. Creative tasks nbenefit from randomness since there may be many acceptable answers, and variety is encouraged.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
#================================================================================================================#
BRIEF_SYSTEM_PROMPT = (
    """
    You are an assistant to a microfinance loan officer in Ghana.
    You support human decision-making.
    You never make the final approve/reject decision yourself.
    Base your analysis only on the letter and extracted data provided.
    Do not invent information.
    """
)

BRIEF_PROMPT = """
Letter:

<<LETTER>>

Extracted data:
<<EXTRACTED_JSON>>

Write a decision-support brief for the loan officer with these sections:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review")

Do NOT recommend "approve" or "reject" — the final decision belongs to the human officer.
"""
#================================================================================================================#

In [ ]:
def generate_brief(letter_id):
    letter_text = LETTERS[letter_id]
    extracted = extract_fields(letter_text)
    extracted_json_str = json.dumps(extracted, indent=2)

    prompt = BRIEF_PROMPT.replace("<<LETTER>>", letter_text).replace("<<EXTRACTED_JSON>>", extracted_json_str)
    return ask_llm(prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0)

In [ ]:
for letter_id in ["L001", "L002","L003", "L006"]:
    print(f"=====Brief for {letter_id}=====")
    print(generate_brief(letter_id))
    print()

=====Brief for L001=====
**Decision-Support Brief**

**1. Strengths**
* **Established business:** The applicant has been operating her provision stall at Makola Market for 12 years.
* **Strong savings history:** She has saved GHS 2,500 through the institution's susu scheme over the past two years with a record of never missing a contribution.
* **Guarantor secured:** She has a guarantor (her sister, who is a teacher) in place.
* **Clear loan purpose:** The funds are intended for business expansion (buying a deep freezer to sell frozen foods).

**2. Risks / red flags**
* **Cash flow tightness:** Her proposed monthly repayment is GHS 450, while her current monthly profit is GHS 900. This means the repayment takes up 50% of her stated current monthly profit, leaving a relatively small margin (GHS 450) for living expenses and business overhead before the expansion generates additional income.
* **Unverified financials:** The stated profit of GHS 900 is self-reported and lacks supporting do

**STUDENT REASONING**

1. Yes, for L003, the model was correctly able to identify strengths, with its registered businesses, employees, documented revenue, collateral, and clear repayment term. The red flags were minimal, while asking for more documentation.
For L006, the model was able to correctly flag some red flags, with no existing business, multiple unrelated ventures, no collateral, no financial history, and an unrealistic repayment timeline. The contrasting reviews are a testament to the model's ability to distinguish strong from weak applications.


2. a. Practical - The model lacks information that a human loan officer would have at their disposal. Like contact details, references, and document verification. Automated decisions based only on text would be unreliable and would miss context.

   b. Ethical - In this situation, loan decisions contain life-changing consequences. If a model wrongly rejects someone, they ight lose their business, livelihood or home. If it wrongly approves someone, they might fall into debt which they cannot repay. Keeping a human in the decision making loop ensures that responsibility is distributed appropriately and allows for appeals.


**PART 3.4**


In [ ]:
print(repr(SUMMARY_SYSTEM_PROMPT))
print(repr(SUMMARY_PROMPT_V2))
print(repr(EXTRACT_SYSTEM_PROMPT))
print(repr(EXTRACT_PROMPT))
print(repr(BRIEF_SYSTEM_PROMPT))
print(repr(BRIEF_PROMPT))

'You are an assistant to a microfinance loan officer. Summarize loan applications factually and neutrally in 3-4 sentences. Do not invent details not stated in the letter, and do not add your own interpretation or judgment of the applicant.'
'Summarize this loan application:\n\n{letter_text}'
'\nYou are a precise data extraction assistant.\nExtract loan application fields as a single valid JSON object with EXACTLY these key:\napplicant_name (string), amount_ghs (number), purpose (string), monthly_profit_ghs (number or null),\nhas_collateral_or_guarantor (boolean),\nrepayment_months (number or null).\nIf a field is not stated in the letter, use null.\nDo not guess.\nReturn ONLY the JSON object, no other text, no markdown fences.\n'
'\nHere is an example.\n\nLetter: \nDear Sir,\nMy name is Ama Kwofie, I sell fabrics at Kantamanto Market.\nI am requesting GHS 5,000 to restock ahead of Easter.\nI make about GHS 600 profit monthly.\nRepayment will be GHS 300 per month for 18 months.\nMy bro

**Commit hash** -[738de712d119c2536a133c99cea0f5c8ba398ef4]

**PART 4.1**

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
#================================================================================================================#
fields_to_check = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
                    "has_collateral_or_guarantor", "repayment_months"]

comparison = {}

for letter_id, gold_values in GOLD.items():
    row = df[df["letter_id"] == letter_id].iloc[0]
    for field in fields_to_check:
        extracted_val = row[field]
        gold_val = gold_values[field]

        if field == "applicant_name":
            match = str(extracted_val).strip().lower() == str(gold_val).strip().lower()
        else:
            # handle NaN vs None safely
            if pd.isna(extracted_val) and gold_val is None:
                match = True
            else:
                match = extracted_val == gold_val

        comparison.setdefault(field, {})[letter_id] = match

comparison_df = pd.DataFrame(comparison).T
comparison_df["accuracy"] = comparison_df.mean(axis=1)
comparison_df
#================================================================================================================#

,L001,L003,L006,accuracy
applicant_name,True,True,True,1.0
amount_ghs,True,True,True,1.0
purpose,False,False,False,0.0
monthly_profit_ghs,True,True,True,1.0
has_collateral_or_guarantor,True,True,True,1.0
repayment_months,True,True,True,1.0


**PART 4.2**

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
#================================================================================================================#
# TODO 1
import json
from collections import Counter

def run_reliability_test(letter_text, temperature, num_runs=5):
    """
    Run extraction multiple times and check consistency.
    NOTE: extract_fields() uses temperature=0 internally, so we need to temporarily
    modify it or create a version that accepts temperature.
    """
    results = []
    valid_json_count = 0
    parse_failures = 0


    print(f"RELIABILITY TEST - L004 at temperature={temperature}")


    for i in range(num_runs):
        # Create a version that accepts temperature parameter
        prompt = EXTRACT_PROMPT.replace("<<LETTER>>", letter_text)
        raw_output = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=temperature)

        # Parse JSON
        cleaned = raw_output.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            cleaned = cleaned.replace("json", "", 1).strip()

        try:
            result = json.loads(cleaned)
            valid_json_count += 1
            canonical = json.dumps(result, sort_keys=True)
            results.append(canonical)
            print(f"\nRun {i+1}: Valid JSON")
            print(f"  applicant_name: {result.get('applicant_name')}")
            print(f"  amount_ghs: {result.get('amount_ghs')}")
            print(f"  purpose: {result.get('purpose')}")
            print(f"  monthly_profit_ghs: {result.get('monthly_profit_ghs')}")
            print(f"  has_collateral_or_guarantor: {result.get('has_collateral_or_guarantor')}")
            print(f"  repayment_months: {result.get('repayment_months')}")
        except (json.JSONDecodeError, AttributeError):
            parse_failures += 1
            print(f"\nRun {i+1}: PARSE FAILURE")
            print(f"  Raw output (first 100 chars): {raw_output[:100]}")

    # Count unique outputs
    unique_outputs = len(set(results)) if results else 0

    # Summary
    print()
    print(f"SUMMARY:")
    print(f"  Valid JSON: {valid_json_count}/{num_runs}")
    print(f"  Parse failures: {parse_failures}/{num_runs}")
    print(f"  Unique outputs: {unique_outputs}/{num_runs}")

    if unique_outputs == 1 and valid_json_count == num_runs:
        print(f"  Verdict: FULLY CONSISTENT  ")
    elif unique_outputs == 1:
        print(f"  Verdict: CONSISTENT but with parse failures .")
    elif unique_outputs < num_runs:
        print(f"  Verdict: PARTIALLY CONSISTENT .")
    else:
        print(f"  Verdict: INCONSISTENT .")

    return {
        "temperature": temperature,
        "valid_json": valid_json_count,
        "parse_failures": parse_failures,
        "unique_outputs": unique_outputs,
        "total_runs": num_runs,
        "results": results
    }

# Get L004 text
l004_text = LETTERS["L004"]
print("Testing letter L004:")
print(f"\"{l004_text[:150]}...\"\n")

# Run reliability tests
print()
print("PART 4.2: RELIABILITY TESTING")
print()

# Test at temperature=0
results_t0 = run_reliability_test(l004_text, temperature=0.0, num_runs=5)

# Test at temperature=1.0
results_t1 = run_reliability_test(l004_text, temperature=1.0, num_runs=5)

# Create comparison table
print()
print("RELIABILITY COMPARISON TABLE")
print()

comparison_data = {
    "Metric": [
        "Valid JSON Rate",
        "Parse Failure Rate",
        "Unique Outputs",
        "Consistency"
    ],
    "Temperature=0.0": [
        f"{results_t0['valid_json']}/{results_t0['total_runs']}",
        f"{results_t0['parse_failures']}/{results_t0['total_runs']}",
        f"{results_t0['unique_outputs']}/{results_t0['total_runs']}",
        "Fully Consistent  " if results_t0['unique_outputs'] == 1 else "Inconsistent ."
    ],
    "Temperature=1.0": [
        f"{results_t1['valid_json']}/{results_t1['total_runs']}",
        f"{results_t1['parse_failures']}/{results_t1['total_runs']}",
        f"{results_t1['unique_outputs']}/{results_t1['total_runs']}",
        "Fully Consistent  " if results_t1['unique_outputs'] == 1 else
        ("Partially Consistent " if results_t1['unique_outputs'] < results_t1['total_runs'] else "Inconsistent .")
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Additional analysis - show what varied between runs at temperature=1.0
if results_t1['unique_outputs'] > 1:
    print()
    print("ANALYSIS: WHAT VARIED AT TEMPERATURE=1.0")
    print()

    # Show the different outputs
    unique_results = list(set(results_t1['results']))
    for i, result_str in enumerate(unique_results):
        result = json.loads(result_str)
        print(f"\nVariant {i+1}:")
        print(f"  monthly_profit_ghs: {result.get('monthly_profit_ghs')}")
        print(f"  repayment_months: {result.get('repayment_months')}")
        print(f"  has_collateral_or_guarantor: {result.get('has_collateral_or_guarantor')}")

    print("""
Key observations:
1. monthly_profit_ghs - The letter says "around GHS 1,500 in a good month"
   Is that monthly profit or just an optimistic estimate? The model was inconsistent.

2. Some runs may have wrapped output in markdown code fences, causing parse failures.

3. The model's interpretation of ambiguous language varied with temperature.
""")
#================================================================================================================#



#================================================================================================================#


Testing letter L004:
"Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm ..."


PART 4.2: RELIABILITY TESTING

RELIABILITY TEST - L004 at temperature=0.0

Run 1: Valid JSON
  applicant_name: Yaw Owusu
  amount_ghs: 12000
  purpose: feed and 500 new layers for poultry farm
  monthly_profit_ghs: 1500
  has_collateral_or_guarantor: True
  repayment_months: 18

Run 2: Valid JSON
  applicant_name: Yaw Owusu
  amount_ghs: 12000
  purpose: feed and 500 new layers for poultry farm
  monthly_profit_ghs: 1500
  has_collateral_or_guarantor: True
  repayment_months: 18

Run 3: Valid JSON
  applicant_name: Yaw Owusu
  amount_ghs: 12000
  purpose: feed and 500 new layers for poultry farm
  monthly_profit_ghs: 1500
  has_collateral_or_guarantor: True
  repayment_months: 18

Run 4: Valid JSON
  applicant_name: Yaw Owusu
  amount_ghs: 12000
  purpose: feed and 500 new layers for poultry farm
  monthly_pro

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
#================================================================================================================#
# TODO 1
print()
print("HALLUCINATION PROBING")
#================================================================================================================#
# TEST 1: Ask about information NOT in the letter
#================================================================================================================#
print()
print("TEST 1: Asking for non-existent information")
print()

letter_text = LETTERS["L001"]  # Akosua's letter
question = "What is the applicant's credit score?"

test1_system_prompt = """You are an assistant to a microfinance loan officer.
Your job is to answer questions about loan application letters.
IMPORTANT: Only use information that is explicitly stated in the letter.
If the information is not present, say 'This information is not provided in the application letter.'
Never guess, estimate, or fabricate information."""

test1_user_prompt = f"""Here is a loan application letter:

{letter_text}

Question: {question}"""

print(f"Letter: L001 (Akosua Mensah)")
print(f"Question: {question}")
print(f"\nModel Response:")
print()

test1_response = ask_llm(test1_user_prompt, system_prompt=test1_system_prompt, temperature=0.0)

print(test1_response)
print()

# Check if model admitted information was absent
admitted_absence = any(phrase in test1_response.lower() for phrase in [
    "not provided", "not mentioned", "not stated", "no information",
    "not available", "does not mention", "does not state", "does not include"
])

invented_score = False
# Check if model invented a credit score (contains a number that could be a credit score)
if any(char.isdigit() for char in test1_response):
    # Check if the number appears to be a credit score (typically 300-850)
    import re
    numbers = re.findall(r'\d{3}', test1_response)
    for num in numbers:
        if 300 <= int(num) <= 850:
            invented_score = True
            break

if admitted_absence and not invented_score:
    print("\n PASS: Model correctly admitted the information is absent")
elif admitted_absence:
    print("\n PARTIAL PASS: Model admitted absence but may have mentioned a number")
else:
    print("\n FAIL: Model may have hallucinated the information")

#================================================================================================================#
# TEST 2: Feed irrelevant text to extractor
#================================================================================================================#
print()
print("TEST 2: Feeding irrelevant text (weather report) to extractor")
print()

irrelevant_text = """Weather Forecast for Accra - Tuesday, August 12, 2026

Morning: Partly cloudy with a 30% chance of showers. Temperature: 27°C.
Afternoon: Sunny intervals with high humidity. Temperature: 31°C.
Evening: Clear skies expected. Temperature: 28°C.

Sunrise: 5:52 AM
Sunset: 6:15 PM
Wind: 12 km/h from the southwest.

Tomorrow's outlook: Continued warm conditions with scattered thunderstorms possible in the late afternoon."""

print("Input text (weather report):")
print(irrelevant_text[:200] + "...\n")
print("Extraction Result:")
print()

# Try to extract loan application fields from weather report
test2_prompt = EXTRACT_PROMPT.replace("<<LETTER>>", irrelevant_text)
test2_raw = ask_llm(test2_prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=0)

print(f"Raw model output:\n{test2_raw}")
print()

# Try to parse
cleaned = test2_raw.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.strip("`")
    cleaned = cleaned.replace("json", "", 1).strip()

try:
    test2_result = json.loads(cleaned)
    print("\nParsed JSON:")
    print(json.dumps(test2_result, indent=2))

    # Check if fields are null or fabricated
    fabricated_fields = []

    if test2_result.get("applicant_name") and test2_result["applicant_name"] != "null":
        fabricated_fields.append(f"applicant_name: '{test2_result['applicant_name']}'")

    if test2_result.get("amount_ghs") and test2_result["amount_ghs"] != 0:
        fabricated_fields.append(f"amount_ghs: {test2_result['amount_ghs']}")

    if test2_result.get("purpose") and test2_result["purpose"] != "null":
        fabricated_fields.append(f"purpose: '{test2_result['purpose']}'")

    if test2_result.get("monthly_profit_ghs") and test2_result["monthly_profit_ghs"] != 0:
        fabricated_fields.append(f"monthly_profit_ghs: {test2_result['monthly_profit_ghs']}")

    if test2_result.get("has_collateral_or_guarantor") is not None and test2_result["has_collateral_or_guarantor"] is not False:
        fabricated_fields.append(f"has_collateral_or_guarantor: {test2_result['has_collateral_or_guarantor']}")

    if test2_result.get("repayment_months") and test2_result["repayment_months"] != 0:
        fabricated_fields.append(f"repayment_months: {test2_result['repayment_months']}")

    if fabricated_fields:
        print("\n FAIL: Model fabricated the following fields:")
        for field in fabricated_fields:
            print(f"  - {field}")
    else:
        print("\n PASS: Model correctly returned null/empty values for all fields")

except (json.JSONDecodeError, AttributeError):
    print("\n PARTIAL PASS: Model did not produce valid JSON")
    print("  (This is actually acceptable for irrelevant input — better than fabricating data)")


#================================================================================================================#


HALLUCINATION PROBING

TEST 1: Asking for non-existent information

Letter: L001 (Akosua Mensah)
Question: What is the applicant's credit score?

Model Response:

This information is not provided in the application letter.


 PASS: Model correctly admitted the information is absent

TEST 2: Feeding irrelevant text (weather report) to extractor

Input text (weather report):
Weather Forecast for Accra - Tuesday, August 12, 2026

Morning: Partly cloudy with a 30% chance of showers. Temperature: 27°C.
Afternoon: Sunny intervals with high humidity. Temperature: 31°C.
Evening...

Extraction Result:

Raw model output:
{
"applicant_name": null,
"amount_ghs": null,
"purpose": null,
"monthly_profit_ghs": null,
"has_collateral_or_guarantor": false,
"repayment_months": null
}


Parsed JSON:
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": null
}

 PASS: Model correctly returned null/emp

**PART 4.4**

If a Bank or a Lending service decide to fully automate their decisions, it would clearly disdvatage the following people:
1. Applicants with poor English skills but run strong businesses.

    The model might flag wrongly worded or poorly articulated text from these people, and flag their applications as incomplete or with no documentation. This often blocks their chances, and these kinds of people are the exact target these microfinance institutions are meant to serve.

2. First-time borrowers.
    People with similar applications to that of L006 are more likely to not be approved if microfinance institutions decide to fully automate their processes. These people often come across as over-ambitious, and for a fully automated system, it completely removes the aid or advice given to these people if a human officer were the one processing their applications.




The implications of sending personal load data to third-party APIs in another country include:
1. The infringement of the Ghana Data Protection Law, which requires user consent before their personal data is processed and also restricts the cross-border transfer of such data.

2. Another could be that it poses both a privacy and a security risk, since personal financial information is shared and can be compromised and intercepted in the face of an attack or poor model training, leaving the data easy for scrapers to access.

before deploying, encryption, dataresidency and consent would be some of the things to cross on the pre-deployment checklist in order to ensure safe use and user consent.

CONCRETE SAFEGUARDS
1. I would ensure human review or a human-centred system, with the system only playing a supporting role. It would provide possible suggestions, probabilities and prediction work.

2.  would also ensure that an appeal can be sent in the first place and the applicant would recieve human help with said appeal, making sure that the model doesnt overlook anyone due to unmatched parameters and requirements.

**PART 5**

**Prompting as engineering**
Prompt iteration and Hyperparameter iteration are similar in the sense that they involve systemati experimentation, where a variable is changed one at a time, as performance is measured, and making sure to keep records of what is working and what is not, to help guide inputs. In lab 3 this was seen through learning rate adjustments and epoch or batch sizes while checking for validation accuracy, while in this lab the prompts and temperatures were experimented (the values) to check for extraction accuracy and consistency.

The main difference is that, Hyperparameters deal with changes in numeric values in a mathematical space with gradient descent, grid search, etc., while prompts deal with natural language and have no defined space, with no gradient to follow. It heavily relies on linguistics, A or B testing among others. For hyperparaeters, working values work accross multiple models since they can be significant but same cannot be siad for prompts since a promt here can be meaningless in another model

**Trust**

I dont think i would trust this system to run unattended. A part of this lab that influenced me to this decision was the hallucination probing and though it was correctly admitted that the credit score was absent, it only did so due to explicit constraints and rules that were stated beforehand. A change in the prompt might result in fabrication of financial information, which is detrimental in a high-stakes environment like this one, where money and the livelihoods of real people are handled.

**Cost and Scale**
For my usage metadata from before with 85 to 100 tokens used, a full pipeline per application would look like this:
1. Summarisation takes about 200 to 300 tokens
2. Extraction, about 100 to 150 tokens
3. Brief generation, 400 to 600 tokens
with the total coming to about 700 to 1050 tokens per loan application.

this number 1000 times(per application) for each month will be around 700,000 to 1,050,000 tokens per month, which is very modest for a free tier model like Gemini's flash lite.
This goes on to imply that:
Cost cannot remain a primary constraint, though it's directly proportional to model performance in some cases.
Free tier remains viable for pilot programs, and provider choice should be influenced by reliability, data privacy, and consistency.

**Looking back at the course**

Calling APIs beats training data models for this task due to the fact that these models have been trained on trillions of tokens and contain vast knowledge. Its also on the cheaper side and handles generalising beautifully.

However training your own model also shines through when data privacy is a concern, when the task is a narrow and well-defined one and is within a highl specialised domain.